In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-07 15:06:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.53MB/s    in 0.2s    

2026-04-07 15:06:02 (6.53 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
len(text)

1115394

In [19]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [16]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(len(chars))

stoi = { c:i for i, c in enumerate(chars) }
itos = { i:c for i, c in enumerate(chars) }
encode = lambda str: [ stoi[c] for c in str]
decode = lambda ii: ''.join([ itos[i] for i in ii ])

print(encode("hello world"))
print(decode(encode("hello world")))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [26]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [61]:
block_size = 8
batch_size = 32
n_embed = 32

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('input', xb.shape, 'output', yb.shape)

input torch.Size([32, 8]) output torch.Size([32, 8])


In [41]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embedding_table(idx) # B,T,C - batch, time, channel
        pos_emb = self.position_embedding_table(torch.arange(T)) # T,C
        x = tok_emb + pos_emb
        logits = self.lm_head(token_emb) # B,T,vocab_size

        if targets is None:
            loss = None
        else:    
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss


    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)      
out, loss = m(xb, None)
print(xb.shape, out.shape, loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8, 65]) None

!JbLjSZ;;bCVN gC Ifqk?blEoC $a!R.wc3t.pWs?xqYfu
!eKPEBjvTBEBjYqEZywgvQ:
qUbgCEPHk?xpzDOouTcHIuJ!cWWu


In [42]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [50]:
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()
    
print(loss.item())

2.4724066257476807


In [52]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


D e aco w; kitowhacarea'lyoucout REEXENGHARKprelroschispis m?
S:

Anon'TEOLUE:
S:
Ing if y, ye geee 


# the math trick in self-attention

In [55]:
B, T, C = 4,8,2
x = torch.randn(B, T, C)
print(x.shape)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)
xbow

torch.Size([4, 8, 2])


tensor([[[ 0.6936, -1.4647],
         [ 0.1814,  0.5556],
         [ 0.5034,  0.4861],
         [ 0.3659,  0.5064],
         [ 0.3564,  0.4826],
         [ 0.1408,  0.4485],
         [ 0.0627,  0.3740],
         [-0.2091,  0.2481]],

        [[-0.3960,  0.3687],
         [ 0.8263, -0.0822],
         [ 0.4023,  0.1228],
         [ 0.1032,  0.0608],
         [ 0.1915,  0.3209],
         [ 0.0486,  0.1640],
         [-0.0095,  0.1552],
         [-0.0648,  0.0416]],

        [[-0.2470,  1.0935],
         [ 0.3348,  0.1098],
         [ 0.3156,  0.0555],
         [ 0.1701,  0.1824],
         [ 0.3270, -0.1771],
         [-0.0466, -0.1413],
         [ 0.0415,  0.0127],
         [ 0.1577, -0.1444]],

        [[ 0.0077,  1.1685],
         [-0.9920,  0.4907],
         [-0.8392,  0.1583],
         [-0.1169,  0.3614],
         [-0.3031,  0.3178],
         [-0.2522,  0.0801],
         [ 0.1123, -0.0670],
         [-0.0568,  0.0336]]])

In [56]:
wei = torch.tril(torch.ones(T, T))
wei = wei/wei.sum(1, keepdim=True)
xbow2 = wei @ x # (T,T) @ (B,T,C) --> (B,T,T) @ (B,T,C) --> (B,T,C)
xbow2

tensor([[[ 0.6936, -1.4647],
         [ 0.1814,  0.5556],
         [ 0.5034,  0.4861],
         [ 0.3659,  0.5064],
         [ 0.3564,  0.4826],
         [ 0.1408,  0.4485],
         [ 0.0627,  0.3740],
         [-0.2091,  0.2481]],

        [[-0.3960,  0.3687],
         [ 0.8263, -0.0822],
         [ 0.4023,  0.1228],
         [ 0.1032,  0.0608],
         [ 0.1915,  0.3209],
         [ 0.0486,  0.1640],
         [-0.0095,  0.1552],
         [-0.0648,  0.0416]],

        [[-0.2470,  1.0935],
         [ 0.3348,  0.1098],
         [ 0.3156,  0.0555],
         [ 0.1701,  0.1824],
         [ 0.3270, -0.1771],
         [-0.0466, -0.1413],
         [ 0.0415,  0.0127],
         [ 0.1577, -0.1444]],

        [[ 0.0077,  1.1685],
         [-0.9920,  0.4907],
         [-0.8392,  0.1583],
         [-0.1169,  0.3614],
         [-0.3031,  0.3178],
         [-0.2522,  0.0801],
         [ 0.1123, -0.0670],
         [-0.0568,  0.0336]]])

In [60]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
xbow3

tensor([[[ 0.6936, -1.4647],
         [ 0.1814,  0.5556],
         [ 0.5034,  0.4861],
         [ 0.3659,  0.5064],
         [ 0.3564,  0.4826],
         [ 0.1408,  0.4485],
         [ 0.0627,  0.3740],
         [-0.2091,  0.2481]],

        [[-0.3960,  0.3687],
         [ 0.8263, -0.0822],
         [ 0.4023,  0.1228],
         [ 0.1032,  0.0608],
         [ 0.1915,  0.3209],
         [ 0.0486,  0.1640],
         [-0.0095,  0.1552],
         [-0.0648,  0.0416]],

        [[-0.2470,  1.0935],
         [ 0.3348,  0.1098],
         [ 0.3156,  0.0555],
         [ 0.1701,  0.1824],
         [ 0.3270, -0.1771],
         [-0.0466, -0.1413],
         [ 0.0415,  0.0127],
         [ 0.1577, -0.1444]],

        [[ 0.0077,  1.1685],
         [-0.9920,  0.4907],
         [-0.8392,  0.1583],
         [-0.1169,  0.3614],
         [-0.3031,  0.3178],
         [-0.2522,  0.0801],
         [ 0.1123, -0.0670],
         [-0.0568,  0.0336]]])

In [70]:
# self attention

B,T,C = 4,8,32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)  
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
v = value(x)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T) 

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v

out

tensor([[[-9.9061e-02,  1.8636e-01,  1.1585e-01, -8.7837e-02, -4.1564e-01,
           3.8376e-01, -1.4274e-01,  1.1007e-01,  4.7567e-01,  2.1975e-01,
           7.3489e-01,  3.5111e-01, -6.4211e-01, -7.4078e-03,  7.9375e-02,
           4.0146e-02],
         [-5.6675e-02,  3.4316e-02,  8.0275e-02,  3.2330e-02, -3.2878e-01,
           8.3410e-02,  7.1007e-03,  1.3474e-01,  2.8703e-01,  2.6721e-01,
           2.5936e-01,  3.3874e-01, -3.0162e-01,  9.4360e-02, -1.9063e-02,
           1.6116e-01],
         [-3.1030e-02, -1.1228e-01,  2.7800e-01,  3.4575e-01, -3.6590e-01,
          -1.7870e-01,  1.6067e-01,  6.2836e-02,  3.2869e-01,  2.6456e-01,
           1.6537e-01,  3.6724e-01, -1.0514e-01, -2.6288e-01,  2.0545e-01,
           5.0722e-01],
         [ 5.0744e-01, -3.9044e-01, -3.1303e-01, -8.2435e-02, -1.1571e-01,
           1.0836e-01,  2.4794e-01, -1.0245e+00,  1.3721e-01,  8.5153e-01,
          -5.4295e-01, -4.0545e-01,  9.9205e-01,  7.4909e-01, -6.9653e-01,
          -3.6182e-01],
    

In [108]:
block_size = 256
batch_size = 64
n_embed = 384
n_layer = 6
n_head = 6
dropout = 0.2

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        wei = q @ k.transpose(-2, -1) * C**-0.5               # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v # (B, T, head_size)

        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        self.sa_heads = MultiHeadAttention(n_head, n_embed//n_head)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)        

    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size) 
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)

    def forward(self, idx, targets=None):
        B, T = idx.shape                                         # (B,T)
        
        tok_emb = self.token_embedding_table(idx)                # (B,T,C) - batch, time, channel
        pos_emb = self.position_embedding_table(torch.arange(T)) # (T,C)
        x = tok_emb + pos_emb                                    # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                 # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:    
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss


    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # crop the idx to block size avoid running out of index in position encoding
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()        

In [109]:
eval_iters = 200
lr = 3e-4

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
for step in range(10000):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 1000 == 0:
        losses = estimate_loss()
        print(f'step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}')
    
print(loss.item())

KeyboardInterrupt: 

In [104]:
print(decode(model.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))        


Badisdard,
Blank not;
Nor they?

KING VAU:
Which flay 'nevery were: Belase,
Her,
there, canshince
LT
